In [ ]:

import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns

In [19]:
from TableToDict import GetDFv2, GetDF, DisplayColumns
from load_ccsb_database import GetOrfMapping
from load_edgotyping_data import load_qY2H

In [ ]:
# COULDN'T IMPORT THESE BUT ALSO NOT USED
from PandasToMySQL import MakeSummaryTable
from HuRI import GetHuRI
#from ccsblib import  huri

In [ ]:
inputs_dir = "../1_inputs"
allele_collection_inputs = "../../1_allele_collection/1_inputs"

# REPLACE THIS WITH TONG'S FILE ONCE HE ADD MUTATION ID
allele_info_df = pd.read_csv(f"{inputs_dir}/VUS_wt_allele_collection_entry_plates_allele_infos.csv", index_col=0)

In [ ]:
allele_info_d = {(orf_id,mutation_id):(nt_change,aa_change,'to_be_verified') for orf_id,mutation_id,nt_change,aa_change 
                 in zip(allele_info_df.orf_id, allele_info_df.mutation_id, allele_info_df.ensg, allele_info_df.nt_change)
                 if mutation_id == mutation_id
                }

allele_collection = GetDFv2()

In [ ]:
for i, (orf_id,nt_change,aa_change,symbol,ensg,entrez,rc4_mut_id,cegs_mut_id,vus_mut_id,rc4_plate,rc4_well,cegs_plate,cegs_well,vus_plate,vus_well,chr,chr_pos_37,chr_pos_38,cv_allele_id,hgmd_id,cv_clinical_significance,cv_phenotypes,cv_phenotype_ids,hgmd_clinical_significance,hgmd_phenotype,clinical_significance) in allele_collection.iterrows():
    for mut_id in [rc4_mut_id,cegs_mut_id,vus_mut_id]:
        if mut_id == mut_id:
            allele_info_d[orf_id,mut_id] = (nt_change,aa_change,clinical_significance)
            
orf_mapping = GetOrfMapping(orf_ids=allele_info_df.orf_id.unique().tolist())
orf_to_symbol = dict(zip(orf_mapping.orf_id, orf_mapping.symbol))



In [ ]:
# NEED TO FIGURE OUT HOW THIS FILE WAS COMPUTED
# IT HAS THE EXPERIMENTAL RESULTS
# NOT GETTING THE RESULTS, JUST THE WELL/PLATE/MUTATION METADATA
# THE RAW CHERRY PICKING FILE SHOULD BE THE REAL INPUT
pdest_layout_df = pd.read_csv(f'{inputs_dir}/VUS_PDEST_dual_experiment_table_annotated.csv', index_col=0)
col = ['exp_plate', 'exp_well', 'orf_id', 'mut_id', 'node_type']
pdest_layout_df = pdest_layout_df[col]
pdest_layout_df['dest_pla_id'] = pdest_layout_df.exp_plate.apply(lambda x: int(x.split('_')[-1])) 

# ran once, did the analysis. some didn't pass the qa/qc thresholds or got no data.
# this is a re-run for all alleles that failed in the first time
# the two batches should be analyzed separately (ie compare allele to the wt on the same batch and plate)

# THIS IS CHERRY PICKING FILE TO LINK RAW DATA FILE TO ORF AND MUT IDs
pdest_layout_b5_df = pd.read_csv(f'{inputs_dir}/summary_CP_VUSDUALXP_batch5.csv', index_col=0)
col = ['orf_id', 'mut_id', 'node_type','dest_pla_id', 'dest_pla', 'dest_well']
pdest_layout_b5_df = pdest_layout_b5_df[col]
pdest_layout_b5_df.rename(columns = {'dest_pla':'exp_plate', 'dest_well':'exp_well'}, inplace=True)

In [ ]:
pdest_layout_df = pd.concat([pdest_layout_b5_df, pdest_layout_df])
# CAN GET THESE FROM TONG'S FILE
pdest_layout_df['nt_change'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][0] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]
pdest_layout_df['aa_change'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][1] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]
# RE-DO
pdest_layout_df['clinical_significance'] = ['WT' if node_type == 'wt' else allele_info_d[orf_id,mut_id][2] if (orf_id,mut_id) in allele_info_d else np.nan for orf_id,mut_id,node_type in zip(pdest_layout_df.orf_id, pdest_layout_df.mut_id, pdest_layout_df.node_type)]

In [ ]:
## Don't have "orf_to_symbol" because I need credentials to access a SQL instance
pdest_layout_df['symbol'] = pdest_layout_df.orf_id.map(orf_to_symbol)

In [7]:
pdest_layout_df.to_csv('pdest_experimental_layout.csv', index=None)

- Maybe we need mutation id in Tong's file (sometimes there are multiple, do we need this?) If we have orf ID and nucleotide change, then you can trace back but it's complicated
- Do we need chr_pos_37 in the allele info file? we only have 38

- VUS_wt_allele_collection_entry_plates_allele_infos.csv can be replaced be the allele_info inputs

CP_VUSDUALXP -> these files map FACS well to ORF id and mutation id
